<a href="https://colab.research.google.com/github/manshikumarics24-hue/BIS-Lab/blob/main/cuckoo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import random
import math

# Sphere Function (Objective Function)
def fitness(x):
    return sum(i * i for i in x)


# Levy Flight using Mantegna's algorithm
def levy_flight():
    beta = 1.5

    sigma = (
        math.gamma(1 + beta) *
        math.sin(math.pi * beta / 2) /
        (
            math.gamma((1 + beta) / 2) *
            beta *
            (2 ** ((beta - 1) / 2))
        )
    ) ** (1 / beta)

    u = random.gauss(0, sigma)
    v = random.gauss(0, 1)

    return u / (abs(v) ** (1 / beta) + 1e-10)


# Main Cuckoo Search Algorithm
def cuckoo_search():

    # User inputs
    N = int(input("Enter number of nests: "))
    T = int(input("Enter number of iterations: "))
    D = int(input("Enter number of dimensions: "))
    Pa = float(input("Enter discovery probability (0 to 1): "))
    lower = float(input("Enter lower bound: "))
    upper = float(input("Enter upper bound: "))

    if N < 2 or T < 1 or D < 1:
        print("Invalid input!")
        return

    if not (0 <= Pa <= 1) or lower >= upper:
        print("Invalid input!")
        return

    alpha = 0.01

    # Step 1: Initialize nests randomly
    nests = [
        [random.uniform(lower, upper) for _ in range(D)]
        for _ in range(N)
    ]

    scores = [fitness(nest) for nest in nests]

    # Step 2: Find best nest
    best_index = scores.index(min(scores))
    best_nest = nests[best_index][:]
    best_score = scores[best_index]

    # Step 3: Main iteration
    for iteration in range(T):

        # Generate new solutions using Levy Flight
        for i in range(N):

            new_nest = nests[i][:]

            for j in range(D):
                new_nest[j] += alpha * levy_flight()

                # Apply boundaries
                new_nest[j] = max(
                    lower, min(upper, new_nest[j])
                )

            new_score = fitness(new_nest)

            # Select random nest
            k = random.randrange(N)

            # Replace if new solution is better
            if new_score < scores[k]:
                nests[k] = new_nest
                scores[k] = new_score

        # Step 4: Abandon nests based on Pa
        for i in range(N):

            if random.random() < Pa:

                nests[i] = [
                    random.uniform(lower, upper)
                    for _ in range(D)
                ]

                scores[i] = fitness(nests[i])

        # Step 5: Update best solution
        best_index = scores.index(min(scores))

        if scores[best_index] < best_score:
            best_score = scores[best_index]
            best_nest = nests[best_index][:]

    # Step 6: Display result
    print("\n----- CUCKOO SEARCH RESULT -----")
    print("Best Solution:", best_nest)
    print("Minimum Fitness Value:", best_score)


# Execute program
cuckoo_search()


Enter number of nests: 20
Enter number of iterations: 100
Enter number of dimensions: 2
Enter discovery probability (0 to 1): 0.25
Enter lower bound: -10
Enter upper bound: 10

----- CUCKOO SEARCH RESULT -----
Best Solution: [0.0010915597020091355, 0.0016534351698904715]
Minimum Fitness Value: 3.925350444081005e-06
